# 03 — DeepEval Results Walkthrough

Reads `outputs/eval_report.json` and walks through each required metric
(FR-J3), the custom routing accuracy (FR-J4), and any failures — with an
honest comment per metric for the final report.

**Run `python tests/evaluate.py` first.**

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.settings import settings

report = json.loads(settings.eval_report_path.read_text(encoding="utf-8"))
print("generated:", report["generated_at"])
print("judge:", report["judge_model"], "| cases:", report["num_cases"])

## Metric summary

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame(report["metrics"]).T
metrics_df

In [ ]:
ra = report["routing_accuracy"]
print(f"Routing accuracy: {ra['correct']}/{ra['total']} = {ra['overall']:.0%}")

rows = []
for c in report["cases"]:
    r = c["routing"]
    rows.append(
        {
            "id": c["id"],
            "lang": c["language"],
            "predicted": r["predicted_domain"],
            "expected": r["expected_domain"],
            "conf": r["confidence"],
            "fallback": r["fallback_applied"],
            "pass": r["pass"],
        }
    )
pd.DataFrame(rows)

## Per-case detail

In [ ]:
rows = []
for c in report["cases"]:
    row = {"id": c["id"], "lang": c["language"]}
    for name, s in c["scores"].items():
        row[name] = s["score"]
    rows.append(row)
pd.DataFrame(rows).set_index("id")

## Failures & judge reasoning

Any case with a metric below threshold — read the judge's stated reason, then
decide whether it's a retrieval miss (fix chunking / top-k), a routing miss
(fix the router prompt), or an answer miss (fix the answer prompt).

In [ ]:
for c in report["cases"]:
    failed = {k: v for k, v in c["scores"].items() if not v["pass"]}
    if not failed and c["routing"]["pass"]:
        continue
    print(f"\n### {c['id']} ({c['language']})")
    print("Q:", c["input"])
    if not c["routing"]["pass"]:
        print(
            f"ROUTING FAIL: predicted={c['routing']['predicted_domain']} "
            f"expected={c['routing']['expected_domain']} conf={c['routing']['confidence']}"
        )
    for name, s in failed.items():
        print(f"  {name}: {s['score']:.3f} — {s['reason'][:300]}")

## Source citation check (FR-G2)

Every predicted source must be a real filename in `data/docs/` — an invented
filename is an immediate fail.

In [ ]:
real = {p.name for d in settings.domains for p in (settings.docs_dir / d).iterdir()}
invented = []
for c in report["cases"]:
    for src in c["predicted_sources"]:
        # sources may carry a trailing ':' or prose — compare exact names only
        if src not in real and not any(name in src for name in real):
            invented.append((c["id"], src))
if invented:
    print("⚠ INVENTED SOURCES (FR-G2 violation):")
    for cid, src in invented:
        print(f"  {cid}: {src}")
else:
    print("✓ Every cited source maps to a real file in data/docs/ (FR-G2)")